<a href="https://colab.research.google.com/github/ranimiftah12345678/ML_22_Ranimiftah/blob/main/JS09/JS09_TUGAS01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q annoy faiss-cpu hnswlib pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np, time, pandas as pd
from annoy import AnnoyIndex
import faiss, hnswlib

np.random.seed(42)

K = 10           # jumlah tetangga
N_QUERY = 100    # jumlah query, hasil dirata-rata
CONFIGS = [(1_000, 2), (1_000, 5), (1_000_000, 2), (1_000_000, 5)]
METRICS = ["euclidean", "cosine"]

def recall(result, truth):
    return np.mean([len(set(r) & set(t)) / K for r, t in zip(result, truth)])

def run(n, dim, metric):
    X = np.random.random((n, dim)).astype(np.float32)
    Q = np.random.random((N_QUERY, dim)).astype(np.float32)
    out = {}

    # ---------- Data untuk FAISS / ground truth ----------
    if metric == "cosine":
        Xf, Qf = X.copy(), Q.copy()
        faiss.normalize_L2(Xf); faiss.normalize_L2(Qf)
        make_flat = lambda: faiss.IndexFlatIP(dim)
    else:
        Xf, Qf = X, Q
        make_flat = lambda: faiss.IndexFlatL2(dim)

    # ---------- FAISS (Flat = exact search, juga jadi ground truth) ----------
    t = time.time()
    fi = make_flat(); fi.add(Xf)
    b = time.time() - t
    t = time.time()
    res = [fi.search(Qf[i:i+1], K)[1][0] for i in range(N_QUERY)]
    q = (time.time() - t) / N_QUERY
    truth = res
    out["FAISS"] = (b, q, recall(res, truth))

    # ---------- ANNOY ----------
    t = time.time()
    ai = AnnoyIndex(dim, "euclidean" if metric == "euclidean" else "angular")
    for i in range(n):
        ai.add_item(i, X[i])
    ai.build(10)
    b = time.time() - t
    t = time.time()
    res = [ai.get_nns_by_vector(Q[i], K) for i in range(N_QUERY)]
    q = (time.time() - t) / N_QUERY
    out["ANNOY"] = (b, q, recall(res, truth))

    # ---------- HNSW ----------
    t = time.time()
    hi = hnswlib.Index(space="l2" if metric == "euclidean" else "cosine", dim=dim)
    hi.init_index(max_elements=n, ef_construction=200, M=16)
    hi.add_items(X)
    b = time.time() - t
    hi.set_ef(50)
    t = time.time()
    res = [hi.knn_query(Q[i:i+1], k=K)[0][0] for i in range(N_QUERY)]
    q = (time.time() - t) / N_QUERY
    out["HNSW"] = (b, q, recall(res, truth))
    return out

rows = []
for n, dim in CONFIGS:
    for metric in METRICS:
        print(f"Menjalankan: {n} data / {dim}D / {metric} ...")
        r = run(n, dim, metric)
        for algo, (b, q, rc) in r.items():
            rows.append(dict(Config=f"{n}/{dim}D", Metric=metric, Algo=algo,
                             Build_s=b, Query_ms=q * 1000, Recall=rc))

df = pd.DataFrame(rows)

def tabel(kolom, judul, fmt):
    print(f"\n=== {judul} ===")
    p = df.pivot_table(index=["Config", "Metric"], columns="Algo", values=kolom, sort=False)
    print(p[["ANNOY", "FAISS", "HNSW"]].round(fmt).to_string())

tabel("Build_s", "BUILD TIME (detik)", 4)
tabel("Query_ms", "QUERY TIME rata-rata per query (ms)", 4)
tabel("Recall", "RECALL@10 (terhadap exact search)", 3)

Menjalankan: 1000 data / 2D / euclidean ...
Menjalankan: 1000 data / 2D / cosine ...
Menjalankan: 1000 data / 5D / euclidean ...
Menjalankan: 1000 data / 5D / cosine ...
Menjalankan: 1000000 data / 2D / euclidean ...
Menjalankan: 1000000 data / 2D / cosine ...
Menjalankan: 1000000 data / 5D / euclidean ...
Menjalankan: 1000000 data / 5D / cosine ...

=== BUILD TIME (detik) ===
Algo                    ANNOY   FAISS      HNSW
Config     Metric                              
1000/2D    euclidean   0.0119  0.0001    0.0260
           cosine      0.0170  0.0000    0.0206
1000/5D    euclidean   0.0080  0.0000    0.0470
           cosine      0.0124  0.0000    0.0356
1000000/2D euclidean  16.4367  0.0023   74.9027
           cosine     42.3837  0.0023  108.9229
1000000/5D euclidean  11.2839  0.0073  118.3584
           cosine     14.6597  0.0077  113.8200

=== QUERY TIME rata-rata per query (ms) ===
Algo                   ANNOY   FAISS    HNSW
Config     Metric                           
1000/

In [3]:
# Cek: apakah tetangga hasil HNSW sebenarnya hampir sama dekatnya dengan exact?
n, dim = 1_000_000, 2
X = np.random.random((n, dim)).astype(np.float32)
Q = np.random.random((N_QUERY, dim)).astype(np.float32)

Xn, Qn = X.copy(), Q.copy()
faiss.normalize_L2(Xn); faiss.normalize_L2(Qn)
gt = faiss.IndexFlatIP(dim); gt.add(Xn)
_, truth = gt.search(Qn, K)

hi = hnswlib.Index(space="cosine", dim=dim)
hi.init_index(max_elements=n, ef_construction=200, M=16)
hi.add_items(X); hi.set_ef(50)
labels, _ = hi.knn_query(Q, k=K)

# pada 2D, cosine hanya bergantung pada sudut -> hitung sudut (float64)
theta  = np.degrees(np.arctan2(X[:, 1].astype(np.float64), X[:, 0].astype(np.float64)))
thetaq = np.degrees(np.arctan2(Q[:, 1].astype(np.float64), Q[:, 0].astype(np.float64)))

gap_ann  = np.mean([np.abs(theta[labels[i]] - thetaq[i]).max() for i in range(N_QUERY)])
gap_true = np.mean([np.abs(theta[truth[i]]  - thetaq[i]).max() for i in range(N_QUERY)])
print("Rata-rata sudut tetangga ke-10 (HNSW)  :", gap_ann,  "derajat")
print("Rata-rata sudut tetangga ke-10 (exact) :", gap_true, "derajat")

Rata-rata sudut tetangga ke-10 (HNSW)  : 0.01508731331112353 derajat
Rata-rata sudut tetangga ke-10 (exact) : 0.0160185591449975 derajat


## Analisis dan Kesimpulan

**1. Pengaruh jumlah data (1.000 vs 1.000.000).**
Pada 1.000 data, waktu query ketiga algoritma hampir sama (sekitar 0,01-0,02 ms), sehingga ANN belum memberi keuntungan berarti. Pada 1.000.000 data, query FAISS Flat melonjak menjadi 1,6-6,4 ms karena membandingkan query dengan seluruh data (brute-force, O(n)). ANNOY (sekitar 0,05 ms) dan HNSW (0,03-0,075 ms) hampir tidak terpengaruh, sehingga keduanya sekitar 100x lebih cepat daripada FAISS Flat pada metrik Euclidean.

**2. Waktu build.**
FAISS Flat paling cepat dibangun (di bawah 0,01 detik, bahkan untuk 1.000.000 data) karena hanya menyimpan vektor. ANNOY membutuhkan 11-42 detik dan HNSW paling lama, yaitu 75-118 detik, karena harus membangun graf berlapis. Ada trade-off yang jelas: build yang mahal dibayar dengan query yang sangat cepat.

**3. Pengaruh dimensi (2D vs 5D).**
Pada 1.000 data, perbedaan 2D dan 5D kecil. Pada 1.000.000 data, build HNSW Euclidean naik dari 74,9 detik (2D) menjadi 118,4 detik (5D) karena perhitungan jarak dan penelusuran graf lebih berat pada dimensi lebih tinggi. Recall ANNOY Euclidean sedikit turun pada 5D (0,944) dibanding 2D (0,984).

**4. Pengaruh metrik jarak (Euclidean vs Cosine).**
Pada kebanyakan konfigurasi, kedua metrik memberi performa yang mirip. Pengecualian terjadi pada 1.000.000 data 2D dengan Cosine: build ANNOY lebih lama (42,4 detik dibanding 16,4 detik) dan recall turun drastis (ANNOY 0,072 dan HNSW 0,266). Pengecekan sudut tetangga ke-10 menunjukkan hasil HNSW (0,0151°) hampir sama dengan exact search (0,0160°), jadi tetangga yang ditemukan sebenarnya sama dekatnya. Penyebabnya adalah keterbatasan presisi float32: pada sudut kecil, cos θ ≈ 1 − θ²/2 sehingga sudut di bawah sekitar 0,02° tidak dapat dibedakan. Akibatnya sekitar 350 titik di sekitar query memiliki nilai cosine identik, dan pemilihan top-10 di antara kandidat yang seri menjadi tidak stabil. Rendahnya recall pada kasus ini adalah artefak evaluasi berbasis ID, bukan kegagalan algoritma. Pada 5D, titik lebih renggang sehingga recall kembali tinggi (ANNOY 0,978 dan HNSW 1,000).

**5. Akurasi (recall).**
FAISS Flat selalu 1,0 karena exact search. HNSW mencapai recall 1,000 pada hampir semua konfigurasi (kecuali kasus Cosine 2D 1.000.000 data di atas). ANNOY berada di 0,94-0,998 dengan 10 pohon, dan dapat ditingkatkan dengan menambah jumlah pohon atau `search_k` dengan konsekuensi build dan query lebih lama.

## 9. Kesimpulan
- **FAISS (Flat)**: akurat 100% dan build tercepat, tetapi query melambat linear terhadap jumlah data. Cocok untuk data kecil atau saat akurasi mutlak diperlukan.
- **ANNOY**: build sedang (11-42 detik untuk 1 juta data), query sangat cepat (sekitar 0,05 ms), recall 0,94-0,998. Kompromi yang baik antara kecepatan dan biaya build.
- **HNSW**: query tercepat dengan recall tertinggi, tetapi build paling mahal (75-118 detik) dan memori lebih besar. Cocok untuk data besar dengan banyak query.
- Pada 1.000 data semua algoritma setara, sehingga FAISS Flat sudah cukup. Keunggulan ANN baru terasa pada data besar.
- Evaluasi recall berbasis ID dapat menyesatkan ketika banyak kandidat memiliki jarak hampir sama (seri) akibat keterbatasan presisi float32, seperti pada Cosine 2D dengan 1.000.000 data. Pada kasus seperti ini, evaluasi sebaiknya mempertimbangkan selisih jarak, bukan hanya kecocokan ID.
- Pemilihan algoritma ditentukan oleh trade-off antara **akurasi, kecepatan query, dan biaya build** sesuai kebutuhan aplikasi.